# LeakageLab: Public Reference Solver
Predict adverse events on future synthetic transactions when labels arrive seven days late.

**Run:** use CPU, turn Internet on, then **Run All**. No private dataset, token or paid resource is needed.
This notebook downloads the four permitted CSVs and only the reference solver modules from the
[GitHub source](https://github.com/VietAnh1803/LeakageLab). It fits four baselines, selects on validation,
and writes `submission.csv`. Test ground truth and author data are excluded.

The benchmark is an independent, AI-assisted portfolio project. Data is synthetic; it is not a Kaggle competition.

## 1. Working directory and public source

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = Path('/kaggle/working' if Path('/kaggle/working').exists() else 'artifacts/public-notebook').resolve()
BUNDLE = ROOT / 'solver-input'
RUNTIME = ROOT / 'runtime'
OUTPUT = ROOT / 'results'
RAW = 'https://raw.githubusercontent.com/VietAnh1803/LeakageLab/main/'
os.environ.update(OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')

def download(relative, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(RAW + relative, timeout=60) as response:
        destination.write_bytes(response.read())

for name in ['__init__.py', 'common.py', 'metrics.py', 'solver.py']:
    download('src/leakagelab/' + name, RUNTIME / 'src/leakagelab' / name)
print('CPU runtime ready; loaded four solver modules, no author modules.')

## 2. Verify the input contract
The train and validation files contain mature labels. The future-test file has no target column.
Checksums must match the published bundle manifest before the solver is launched.

In [ ]:
download('public/solver-input/bundle_manifest.json', BUNDLE / 'bundle_manifest.json')
manifest = json.loads((BUNDLE / 'bundle_manifest.json').read_text())
for relative, expected in manifest['files'].items():
    name = Path(relative).name
    assert name in {'train.csv', 'validation.csv', 'test.csv', 'sample_submission.csv'}
    download('public/solver-input/' + name, BUNDLE / name)
    assert hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest() == expected, name

sizes = []
for name in ['train.csv', 'validation.csv', 'test.csv', 'sample_submission.csv']:
    frame = pd.read_csv(BUNDLE / name)
    if name in {'test.csv', 'sample_submission.csv'}:
        assert 'target' not in frame.columns
    sizes.append({'input': name, 'rows': len(frame), 'columns': len(frame.columns)})
display(pd.DataFrame(sizes))

## 3. Fit, select, and predict
The dummy baseline, logistic regression, and two gradient-boosting settings share the same inputs.
Validation average precision selects the model. History ablation and the random-split diagnostic
are excluded from selection; random splitting is not a valid estimate of future deployment.

In [ ]:
env = dict(os.environ, PYTHONPATH=str(RUNTIME / 'src'))
completed = subprocess.run(
    [sys.executable, '-m', 'leakagelab.solver', '--bundle', str(BUNDLE),
     '--output', str(OUTPUT), '--seed', '1803', '--threads', '2'],
    cwd=BUNDLE, env=env, capture_output=True, text=True, timeout=1800,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()
experiments = pd.read_csv(OUTPUT / 'experiments.csv')
display(experiments[['model', 'track', 'history', 'average_precision', 'roc_auc']])

## 4. Inspect the submission

In [ ]:
submission = pd.read_csv(OUTPUT / 'submission.csv')
sample = pd.read_csv(BUNDLE / 'sample_submission.csv')
assert submission.transaction_id.is_unique
assert set(submission.transaction_id) == set(sample.transaction_id)
assert submission.probability.between(0, 1).all()
display(submission.head())
print(f'PUBLIC_SOLVER_COMPLETE: {len(submission):,} predictions saved to {OUTPUT / "submission.csv"}')

## 5. Recorded author-side evaluation
The figures below are the **previously verified author run**, not a new test score from this notebook.
The public solver cannot grade final test because it has no test labels. Validation results above
are freshly computed. Source code includes a synthetic generator, so this is an input-contract
demonstration, not a secret or adversarial holdout.

In [ ]:
download('artifacts/kaggle/results/metrics.json', ROOT / 'recorded-author-metrics.json')
recorded = json.loads((ROOT / 'recorded-author-metrics.json').read_text())
display(pd.Series(recorded['test'], name='Recorded author-side test').to_frame())

## Limits
One synthetic seed and one temporal split do not establish real-world generalization.
Account repetition is intentional; test prevalence differs from validation, so AP across the
two partitions is not directly comparable. The generator is public and can reproduce labels.
No medal, production impact, or Eris acceptance is claimed.